<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-12-production-deploy/lesson-12.3-admin-observability/notebooks/GCP_Capstone_12.3_AdminObservability.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12.3 Admin Dashboard & Observability
**Netsetos GenAI Engineering — GCP Capstone**

Log Sink → BigQuery, Cloud DLP redaction, tenant usage dashboard, audit log viewer, budget SLO alerts. The operability layer every regulated tenant demands.

## Cell 1: Log Sink — structured API logs → BigQuery

In [ ]:
SINK_TF = '''
resource "google_bigquery_dataset" "observability" {
  dataset_id    = "documind_observability"
  location      = var.india_region
  description   = "API structured logs, DLP findings, tenant rollups"
  default_table_expiration_ms = 7776000000   # 90 days for raw logs
  delete_contents_on_destroy  = false
}

resource "google_logging_project_sink" "api_to_bq" {
  name        = "documind-api-to-bq"
  destination = "bigquery.googleapis.com/projects/${var.project_id}/datasets/${google_bigquery_dataset.observability.dataset_id}"
  filter      = <<EOT
    resource.type = "cloud_run_revision"
    resource.labels.service_name = "documind-api"
    jsonPayload.event = "query"
  EOT
  unique_writer_identity = true
  bigquery_options { use_partitioned_tables = true }
}

# Sink identity needs BQ data editor
resource "google_bigquery_dataset_iam_member" "sink_writer" {
  dataset_id = google_bigquery_dataset.observability.dataset_id
  role       = "roles/bigquery.dataEditor"
  member     = google_logging_project_sink.api_to_bq.writer_identity
}
'''
with open('sink.tf', 'w') as f: f.write(SINK_TF)
print('sink.tf written')
print()
print('Partitioned by day on timestamp. One row per /v1/query call.')
print('Columns auto-derived from jsonPayload: tenant, user, latency_ms, tokens_in/out, confidence, answerable')

## Cell 2: Daily rollup materialised view

In [ ]:
DAILY_ROLLUP_SQL = '''
CREATE MATERIALIZED VIEW IF NOT EXISTS `documind_observability.tenant_daily`
PARTITION BY day
CLUSTER BY tenant
AS
SELECT
  DATE(timestamp, "Asia/Kolkata") AS day,
  jsonPayload.tenant AS tenant,
  COUNT(*) AS queries,
  COUNTIF(jsonPayload.answerable = false) AS unanswerable,
  SUM(CAST(jsonPayload.tokens_in  AS INT64)) AS tokens_in,
  SUM(CAST(jsonPayload.tokens_out AS INT64)) AS tokens_out,
  APPROX_QUANTILES(CAST(jsonPayload.latency_ms AS INT64), 100)[OFFSET(50)] AS p50_ms,
  APPROX_QUANTILES(CAST(jsonPayload.latency_ms AS INT64), 100)[OFFSET(95)] AS p95_ms,
  APPROX_QUANTILES(CAST(jsonPayload.latency_ms AS INT64), 100)[OFFSET(99)] AS p99_ms
FROM `documind_observability.run_googleapis_com_stdout`
WHERE jsonPayload.event = "query"
GROUP BY day, tenant;
'''
with open('tenant_daily.sql', 'w') as f: f.write(DAILY_ROLLUP_SQL)
print('tenant_daily.sql written')
print()
print('Materialised view refreshes automatically. Query cost for dashboard ~10MB scanned.')

## Cell 3: Cloud DLP — scan uploads for PII (Aadhaar, PAN, email, phone)

In [ ]:
DLP_PY = '''
import os
from google.cloud import dlp_v2
from google.cloud import firestore

PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
INDIA_INFO_TYPES = [
    {"name": "INDIA_AADHAAR_INDIVIDUAL"},
    {"name": "INDIA_PAN_INDIVIDUAL"},
    {"name": "INDIA_GST_INDIVIDUAL"},
    {"name": "EMAIL_ADDRESS"},
    {"name": "PHONE_NUMBER"},
    {"name": "PERSON_NAME"},
    {"name": "DATE_OF_BIRTH"},
]
MIN_LIKELIHOOD = "LIKELY"

_dlp = dlp_v2.DlpServiceClient()
_fs  = firestore.Client()

def inspect_and_log(chunk_id: str, tenant_id: str, text: str) -> dict:
    parent = f"projects/{PROJECT}/locations/asia-south1"
    resp = _dlp.inspect_content(
        request={
            "parent": parent,
            "inspect_config": {
                "info_types": INDIA_INFO_TYPES,
                "min_likelihood": MIN_LIKELIHOOD,
                "include_quote": False,  # DO NOT store PII in audit logs
                "limits": {"max_findings_per_request": 50},
            },
            "item": {"value": text},
        })
    findings = [{"info_type": f.info_type.name, "likelihood": f.likelihood.name,
                 "offset": f.location.byte_range.start if f.location.byte_range else None}
                for f in resp.result.findings]
    if findings:
        _fs.collection("dlp_findings").add({
            "chunk_id": chunk_id, "tenant_id": tenant_id,
            "findings": findings, "count": len(findings),
            "scanned_at": firestore.SERVER_TIMESTAMP,
        })
    return {"has_pii": bool(findings), "types": sorted({f["info_type"] for f in findings})}

def redact(text: str) -> str:
    """Use BEFORE sending to external providers (OpenAI fallback)."""
    parent = f"projects/{PROJECT}/locations/asia-south1"
    resp = _dlp.deidentify_content(request={
        "parent": parent,
        "inspect_config": {"info_types": INDIA_INFO_TYPES, "min_likelihood": "POSSIBLE"},
        "deidentify_config": {"info_type_transformations": {
            "transformations": [{
                "primitive_transformation": {
                    "replace_with_info_type_config": {}
                }}]}},
        "item": {"value": text},
    })
    return resp.item.value  # "Send to [EMAIL_ADDRESS] by [DATE_OF_BIRTH]"
'''
with open('dlp.py', 'w') as f: f.write(DLP_PY)
print('dlp.py written')
print()
print('inspect_and_log() used on every uploaded chunk during ingestion (Module 11 pipeline hook)')
print('redact() used on every prompt routed to non-India providers (LiteLLM pre-request hook)')

## Cell 4: Audit log emitter — every sensitive action, append-only

In [ ]:
AUDIT_PY = '''
import os, uuid, json
from datetime import datetime, timezone
from google.cloud import storage, firestore

AUDIT_BUCKET = storage.Client().bucket(os.environ["AUDIT_BUCKET"])   # 5y LOCKED retention (12.1)
_fs = firestore.Client()

AUDIT_ACTIONS = {
    "user.login", "user.logout",
    "doc.upload", "doc.delete", "doc.download",
    "query.submit", "query.export",
    "admin.view_tenant", "admin.rotate_key",
    "tenant.create", "tenant.suspend",
    "dlp.finding", "consent.grant", "consent.revoke",
}

def emit(action: str, actor: dict, target: dict, meta: dict | None = None):
    assert action in AUDIT_ACTIONS, f"unregistered audit action: {action}"
    event = {
        "id": str(uuid.uuid4()),
        "ts": datetime.now(timezone.utc).isoformat(),
        "action": action,
        "actor": actor,     # {email, sub, tenant_id, ip}
        "target": target,   # {type, id, tenant_id}
        "meta": meta or {},
    }
    # Path: year/month/day/tenant/action-timestamp.json
    d = datetime.now(timezone.utc)
    blob = AUDIT_BUCKET.blob(
        f"{d.year}/{d.month:02d}/{d.day:02d}/{actor.get('tenant_id','_')}/"
        f"{action}-{event['id']}.json")
    blob.upload_from_string(json.dumps(event, separators=(',',':')),
                            content_type="application/json")
    # Also index in Firestore for fast dashboard filter (14-day TTL)
    _fs.collection("audit_index").document(event["id"]).set(event)
    return event["id"]
'''
with open('audit.py', 'w') as f: f.write(AUDIT_PY)
print('audit.py written')
print()
print('Dual-write pattern: GCS is source of truth (5y retention, locked).')
print('Firestore audit_index is a 14-day hot index for the dashboard.')

## Cell 5: admin_dashboard.py — Streamlit admin tabs

In [ ]:
ADMIN_PY = '''
import os, pandas as pd, plotly.express as px, streamlit as st
from google.cloud import bigquery, firestore

_bq = bigquery.Client()
_fs = firestore.Client()
DATASET = "documind_observability"

@st.cache_data(ttl=300)
def tenant_daily(tenant: str | None, days: int = 30) -> pd.DataFrame:
    where = "day >= DATE_SUB(CURRENT_DATE('Asia/Kolkata'), INTERVAL @days DAY)"
    params = [bigquery.ScalarQueryParameter("days", "INT64", days)]
    if tenant:
        where += " AND tenant = @t"
        params.append(bigquery.ScalarQueryParameter("t", "STRING", tenant))
    sql = f"SELECT * FROM `{DATASET}.tenant_daily` WHERE {where} ORDER BY day"
    return _bq.query(sql, job_config=bigquery.QueryJobConfig(query_parameters=params)).to_dataframe()

def usage_tab():
    st.subheader("Usage (last 30 days)")
    tenant = st.selectbox("Tenant filter", ["All"] + list_tenants()) or "All"
    df = tenant_daily(None if tenant == "All" else tenant)
    if df.empty: st.info("No queries in window."); return
    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Total queries", f"{df['queries'].sum():,}")
    col2.metric("Tokens (M)", f"{(df['tokens_in'].sum()+df['tokens_out'].sum())/1e6:.2f}")
    col3.metric("p95 latency", f"{df['p95_ms'].median():.0f} ms")
    unans_pct = 100*df["unanswerable"].sum()/max(1, df["queries"].sum())
    col4.metric("Unanswerable", f"{unans_pct:.1f} %")
    st.plotly_chart(px.bar(df, x="day", y="queries", color="tenant",
                           title="Queries per day"))
    st.plotly_chart(px.line(df, x="day", y=["p50_ms","p95_ms","p99_ms"],
                            title="Latency percentiles (ms)"))

def tenants_tab():
    st.subheader("Tenants")
    rows = []
    for t in _fs.collection("tenants").stream():
        d = t.to_dict(); d["id"] = t.id; rows.append(d)
    df = pd.DataFrame(rows)
    st.dataframe(df, use_container_width=True)
    with st.expander("Create tenant"):
        tid = st.text_input("Tenant ID")
        tier = st.selectbox("Tier", ["free","pro","enterprise"])
        quota = st.number_input("Monthly budget (USD)", 10, 10000, 50)
        if st.button("Create") and tid:
            _fs.collection("tenants").document(tid).set({
                "tier": tier, "max_budget_usd": quota,
                "created_at": firestore.SERVER_TIMESTAMP,
            })
            from audit import emit
            emit("tenant.create", st.session_state.user,
                 {"type":"tenant","id":tid}, {"tier":tier,"budget":quota})
            st.success(f"Created {tid}"); st.rerun()

def audit_tab():
    st.subheader("Audit log (last 14 days)")
    action = st.selectbox("Action", ["all","user.login","doc.upload","doc.delete",
                                     "admin.rotate_key","tenant.suspend"])
    q = _fs.collection("audit_index").order_by("ts", direction="DESCENDING").limit(500)
    if action != "all": q = q.where("action", "==", action)
    rows = [d.to_dict() for d in q.stream()]
    st.dataframe(pd.DataFrame(rows), use_container_width=True)
    st.caption("Full 5-year audit is in GCS. Firestore index is hot 14 days only.")

def dlp_tab():
    st.subheader("DLP findings")
    q = _fs.collection("dlp_findings").order_by("scanned_at", direction="DESCENDING").limit(200)
    rows = [d.to_dict() for d in q.stream()]
    df = pd.DataFrame(rows)
    if df.empty: st.info("No findings."); return
    st.metric("Chunks with findings", len(df))
    flat = []
    for _, r in df.iterrows():
        for f in r["findings"]:
            flat.append({"tenant": r["tenant_id"], "type": f["info_type"],
                         "likelihood": f["likelihood"]})
    st.plotly_chart(px.histogram(pd.DataFrame(flat), x="type", color="likelihood",
                                 title="PII findings by type"))

def list_tenants() -> list[str]:
    return sorted(t.id for t in _fs.collection("tenants").stream())

def admin_page(user):
    st.title("\U0001F6E0 DocuMind Admin")
    tab1, tab2, tab3, tab4 = st.tabs(["Usage","Tenants","Audit Log","DLP"])
    with tab1: usage_tab()
    with tab2: tenants_tab()
    with tab3: audit_tab()
    with tab4: dlp_tab()
'''
with open('admin_dashboard.py', 'w') as f: f.write(ADMIN_PY)
print('admin_dashboard.py written')

## Cell 6: Cloud Monitoring alert policies (SLO + budget)

In [ ]:
ALERTS_TF = '''
resource "google_monitoring_notification_channel" "oncall" {
  display_name = "DocuMind on-call"
  type         = "pagerduty"
  labels       = { service_key = var.pagerduty_key }
  sensitive_labels { auth_token { value = var.pagerduty_key } }
}

# SLO: p95 /v1/query < 3s over 30-min rolling window
resource "google_monitoring_alert_policy" "api_latency" {
  display_name = "API p95 latency > 3s"
  combiner     = "OR"
  conditions {
    display_name = "p95 > 3s for 5 minutes"
    condition_threshold {
      filter     = "resource.type=\\"cloud_run_revision\\" AND resource.labels.service_name=\\"documind-api\\" AND metric.type=\\"run.googleapis.com/request_latencies\\""
      comparison = "COMPARISON_GT"
      threshold_value = 3000
      duration   = "300s"
      aggregations {
        alignment_period     = "60s"
        per_series_aligner   = "ALIGN_PERCENTILE_95"
        cross_series_reducer = "REDUCE_MEAN"
      }
    }
  }
  notification_channels = [google_monitoring_notification_channel.oncall.id]
  alert_strategy { auto_close = "1800s" }
}

# Unanswerable rate spike (product signal, not infra)
resource "google_monitoring_alert_policy" "unanswerable_rate" {
  display_name = "Unanswerable rate > 20% for a tenant"
  combiner     = "OR"
  conditions {
    display_name = "unanswerable_rate > 0.20"
    condition_threshold {
      filter     = "metric.type=\\"logging.googleapis.com/user/documind/unanswerable_rate\\""
      comparison = "COMPARISON_GT"
      threshold_value = 0.20
      duration   = "1800s"
      aggregations {
        alignment_period     = "300s"
        per_series_aligner   = "ALIGN_MEAN"
      }
    }
  }
  notification_channels = [google_monitoring_notification_channel.oncall.id]
}
'''
with open('alerts.tf', 'w') as f: f.write(ALERTS_TF)
print('alerts.tf written')
print()
print('Two policies: infra SLO (p95 latency) + product signal (unanswerable rate).')
print('Product signal catches retrieval regressions that would pass every infra check.')

## Cell 7: Deploy the admin UI + IAM

In [ ]:
DEPLOY = '''
# Admin UI is the SAME Streamlit app as 12.4, but with the Admin tab gated by is_admin(user).
# We deploy a SEPARATE Cloud Run service for admins with additional IAM + IAP policy.

gcloud run deploy documind-admin \\
  --image=us-central1-docker.pkg.dev/$PROJECT/documind/ui:$GIT_SHA \\
  --region=us-central1 --no-allow-unauthenticated \\
  --ingress=internal-and-cloud-load-balancing \\
  --service-account=documind-admin-sa@$PROJECT.iam.gserviceaccount.com \\
  --set-env-vars="ADMIN_EMAILS=alice@documind.ai,bob@documind.ai,AUDIT_BUCKET=$PROJECT-audit" \\
  --set-secrets="COOKIE_SECRET=cookie-secret:latest" \\
  --session-affinity --cpu-boost

# IAP group restricted to admin-group@documind.ai
gcloud beta run services update documind-admin --region=us-central1 --iap
gcloud beta iap web add-iam-policy-binding \\
  --resource-type=cloud-run --service=documind-admin --region=us-central1 \\
  --member="group:admin-group@documind.ai" \\
  --role="roles/iap.httpsResourceAccessor"

# Admin SA needs BQ data viewer + Firestore user
gcloud projects add-iam-policy-binding $PROJECT \\
  --member="serviceAccount:documind-admin-sa@$PROJECT.iam.gserviceaccount.com" \\
  --role="roles/bigquery.dataViewer"
gcloud projects add-iam-policy-binding $PROJECT \\
  --member="serviceAccount:documind-admin-sa@$PROJECT.iam.gserviceaccount.com" \\
  --role="roles/bigquery.jobUser"
'''
print(DEPLOY)

## Cell 8: Smoke tests + what 12.4 inherits

In [ ]:
SMOKE = '''
# 1. Sink is writing
bq query --use_legacy_sql=false \\
  "SELECT COUNT(*) FROM documind_observability.run_googleapis_com_stdout \\
   WHERE timestamp > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 HOUR)"
# -> positive integer = sink healthy

# 2. DLP finds PII in a test doc
python -c "from dlp import inspect_and_log; print(inspect_and_log(\\
  'test_chunk', 'tenant-acme', 'Contact raj@example.in PAN ABCDE1234F'))"
# -> {"has_pii": true, "types": ["EMAIL_ADDRESS","INDIA_PAN_INDIVIDUAL"]}

# 3. Audit emit writes to GCS
python -c "from audit import emit; print(emit(\\
  \"doc.upload\", {\"email\":\"alice@acme.in\",\"tenant_id\":\"tenant-acme\"},\\
  {\"type\":\"doc\",\"id\":\"doc_9f23\"}))"
gsutil ls gs://$PROJECT-audit/$(date +%Y)/$(date +%m)/$(date +%d)/tenant-acme/

# 4. Non-admin user cannot see admin tab
# Open documind-admin Cloud Run URL as a non-admin user -> IAP 403.

# 5. Alert fires on synthetic latency spike
hey -n 200 -c 20 -m POST -H "Authorization: Bearer $TOK" \\
  https://documind-api-xxx.run.app/v1/query
# Wait 6 min, PagerDuty page arrives if p95 > 3000ms
'''
print(SMOKE)
print()
print('WHAT 12.4 (Streamlit UI) INHERITS FROM THIS LESSON:')
print('  - admin_dashboard.py already implements the Admin tab')
print('  - audit.emit() called on doc.upload / doc.delete / query.export in the UI flows')
print('  - dlp.redact() applied before any prompt routed to LiteLLM non-India backend')
print('  - Admins access documind-admin Cloud Run service via IAP group membership')

## ✅ Lesson 12.3 Complete!

- ✅ Log Sink: structured API logs → BigQuery dataset, partitioned, 90-day retention
- ✅ Materialised view `tenant_daily` with tokens + latency percentiles + unanswerable rate
- ✅ Cloud DLP with India info types (Aadhaar, PAN, GST), `inspect_and_log` + `redact`
- ✅ Append-only audit log: GCS 5y locked (source of truth) + Firestore 14d hot index
- ✅ Streamlit admin tabs: Usage / Tenants / Audit / DLP (admin_dashboard.py)
- ✅ Two Cloud Monitoring alert policies: infra SLO + product signal (unanswerable rate)
- ✅ Separate Cloud Run service for admin UI, IAP group gating, admin-sa scoped roles

**Next: Lesson 12.4 — Streamlit Frontend on Cloud Run (the user-facing layer)**